# Build Train / Val / Test Splits

Builds all 36 cell CSVs + 2 RFP window CSVs under `data/splits/` from raw
price series in `data/daily/` and `data/weekly/`, using boundaries defined in
`data/splits/manifest.yaml`.

See `data/splits/explanation.md` for the full splitting strategy.

Run this notebook end-to-end to (re)generate splits. All outputs are
deterministic.


In [1]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import random

ROOT = Path.cwd()
DATA = ROOT / "data"
SPLITS = DATA / "splits"
RAW = {"daily": DATA / "daily", "weekly": DATA / "weekly"}

with open(SPLITS / "manifest.yaml") as f:
    cfg = yaml.safe_load(f)

cfg


{'sample_start': datetime.date(2004, 1, 1),
 'splits': {'daily': {'train': [datetime.date(2004, 1, 1),
    datetime.date(2021, 12, 31)],
   'val': [datetime.date(2022, 1, 1), datetime.date(2023, 12, 31)],
   'test': [datetime.date(2024, 1, 1), datetime.date(2026, 4, 29)]},
  'weekly': {'train': [datetime.date(2004, 1, 2), datetime.date(2018, 12, 31)],
   'val': [datetime.date(2019, 1, 1), datetime.date(2019, 12, 31)],
   'test': [datetime.date(2020, 1, 1), datetime.date(2026, 4, 29)]}},
 'targets': ['SPY', 'OIL', 'GOLD'],
 'exog_pool': ['SPY', 'OIL', 'GOLD', 'VIX', 'DXY'],
 'embargo_periods': 1,
 'rfp': {'regimes': {'GFC': [datetime.date(2007, 7, 1),
    datetime.date(2009, 6, 30)],
   'OIL_CRASH': [datetime.date(2014, 6, 1), datetime.date(2016, 2, 29)],
   'COVID': [datetime.date(2020, 2, 1), datetime.date(2020, 12, 31)],
   'ENERGY_22': [datetime.date(2022, 1, 1), datetime.date(2023, 6, 30)],
   'CALM_17_19': [datetime.date(2017, 1, 1), datetime.date(2019, 12, 31)]},
  'daily': {'n_p

## 1. Load and align raw price series

The five raw weekly CSVs are anchored on different days of the week (SPY/VIX/DXY
on Saturdays, OIL/GOLD on Mondays), so a naive inner join on the date index
yields zero rows. We normalize each weekly date to the **Friday of its ISO
week** before joining, giving a consistent weekly calendar across all series.
Daily series are joined directly on the trading date.


In [2]:
SERIES = ["SPY", "OIL", "GOLD", "VIX", "DXY"]

def load_close(path: Path, freq: str) -> pd.Series:
    df = pd.read_csv(path, skiprows=[1, 2], header=0).rename(columns={"Price": "Date"})
    df["Date"] = pd.to_datetime(df["Date"])
    s = df.set_index("Date")["Close"].astype(float).sort_index()
    if freq == "weekly":
        # normalize to Friday of the ISO week to align cross-series anchors
        s.index = s.index.to_period("W-FRI").to_timestamp(how="end").normalize()
        s = s[~s.index.duplicated(keep="last")]
    return s

def load_panel(freq: str) -> pd.DataFrame:
    closes = {s: load_close(RAW[freq] / f"{s}_{freq}.csv", freq) for s in SERIES}
    panel = pd.concat(closes, axis=1).dropna(how="any")
    panel = panel.loc[panel.index >= pd.Timestamp(cfg["sample_start"])]
    panel.index.name = "date"
    return panel

panels = {freq: load_panel(freq) for freq in ("daily", "weekly")}
for freq, p in panels.items():
    print(f"{freq:6s}  n={len(p):5d}  range={p.index.min().date()} -> {p.index.max().date()}")


daily   n= 5603  range=2004-01-05 -> 2026-04-29
weekly  n= 1166  range=2004-01-02 -> 2026-05-01


/tmp/ipykernel_6298/179856352.py:15: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  panel = pd.concat(closes, axis=1).dropna(how="any")


## 2. Build feature panels

For each frequency:
- `{S}_ret`  = log-return for SPY/OIL/GOLD/DXY
- `vix_log`  = log of VIX level (VIX is already a volatility measure; differencing destroys signal)
- lagged versions: `{S}_ret_lag1`, `vix_log_lag1`


In [3]:
def build_features(panel: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=panel.index)
    for s in ["SPY", "OIL", "GOLD", "DXY"]:
        out[f"{s}_ret"] = np.log(panel[s]).diff()
    out["vix_log"] = np.log(panel["VIX"])
    for col in list(out.columns):
        out[f"{col}_lag1"] = out[col].shift(1)
    return out.dropna()

features = {freq: build_features(p) for freq, p in panels.items()}
for freq, f in features.items():
    print(f"{freq:6s}  n={len(f):5d}  range={f.index.min().date()} -> {f.index.max().date()}")
features["daily"].head(3)


daily   n= 5598  range=2004-01-07 -> 2026-04-29
weekly  n= 1164  range=2004-01-16 -> 2026-05-01


/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,SPY_ret,OIL_ret,GOLD_ret,DXY_ret,vix_log,SPY_ret_lag1,OIL_ret_lag1,GOLD_ret_lag1,DXY_ret_lag1,vix_log_lag1
date,,,,,,,,,,
2004-01-07,0.003371,-0.002377,-0.002131,0.006634,2.740840,0.000978,-0.002371,-0.003777,-0.005125,2.817203
2004-01-08,0.003977,0.010651,0.004965,-0.007335,2.747912,0.003371,-0.002377,-0.002131,0.006634,2.740840
2004-01-09,-0.008770,0.009665,0.005644,-0.005390,2.818398,0.003977,0.010651,0.004965,-0.007335,2.747912


## 3. Build per-cell tables

For each (frequency, exog_flag, target) cell, assemble a `(date, ret, ret_lag1, [exog_lags])` table.
The target series is excluded from its own exogenous set.


In [4]:
def build_cell(freq: str, target: str, use_exog: bool) -> pd.DataFrame:
    f = features[freq]
    cols = {"ret": f[f"{target}_ret"], "ret_lag1": f[f"{target}_ret_lag1"]}
    if use_exog:
        for s in ["SPY", "OIL", "GOLD", "DXY"]:
            if s == target:
                continue
            cols[f"{s}_ret_lag1"] = f[f"{s}_ret_lag1"]
        cols["vix_log_lag1"] = f["vix_log_lag1"]
    df = pd.DataFrame(cols, index=f.index).dropna()
    df.index.name = "date"
    return df.reset_index()

build_cell("daily", "SPY", True).head(3)


,date,ret,ret_lag1,OIL_ret_lag1,GOLD_ret_lag1,DXY_ret_lag1,vix_log_lag1
0,2004-01-07,0.003371,0.000978,-0.002371,-0.003777,-0.005125,2.817203
1,2004-01-08,0.003977,0.003371,-0.002377,-0.002131,0.006634,2.740840
2,2004-01-09,-0.008770,0.003977,0.010651,0.004965,-0.007335,2.747912


## 4. Slice into train / val / test and write CSVs

In [5]:
def slice_stage(df: pd.DataFrame, stage_dates) -> pd.DataFrame:
    start, end = pd.Timestamp(stage_dates[0]), pd.Timestamp(stage_dates[1])
    return df[(df["date"] >= start) & (df["date"] <= end)].reset_index(drop=True)

written = []
for freq in ("daily", "weekly"):
    for use_exog, sub in [(False, "no_exog"), (True, "with_exog")]:
        for target in cfg["targets"]:
            cell = build_cell(freq, target, use_exog)
            for stage, dates in cfg["splits"][freq].items():
                out = slice_stage(cell, dates)
                outdir = SPLITS / freq / sub / target
                outdir.mkdir(parents=True, exist_ok=True)
                out.to_csv(outdir / f"{stage}.csv", index=False, date_format="%Y-%m-%d")
                written.append((freq, sub, target, stage, len(out)))

summary = pd.DataFrame(written, columns=["freq", "exog", "target", "stage", "rows"])
print(f"wrote {len(summary)} split files")
summary.pivot_table(index=["freq", "exog", "target"], columns="stage", values="rows")


wrote 36 split files


stage                     test   train    val
freq   exog      target                      
daily  no_exog   GOLD    583.0  4514.0  501.0
                 OIL     583.0  4514.0  501.0
                 SPY     583.0  4514.0  501.0
       with_exog GOLD    583.0  4514.0  501.0
                 OIL     583.0  4514.0  501.0
                 SPY     583.0  4514.0  501.0
weekly no_exog   GOLD    330.0   781.0   52.0
                 OIL     330.0   781.0   52.0
                 SPY     330.0   781.0   52.0
       with_exog GOLD    330.0   781.0   52.0
                 OIL     330.0   781.0   52.0
                 SPY     330.0   781.0   52.0

## 5. Generate RFP window definitions

For each (frequency, regime), draw up to `n_per_regime` non-overlapping windows
of length `window_len`, with each window's `forecast_end` strictly before the
frequency's `test_start`. Regimes whose entire span lies on/after `test_start`
are skipped for that frequency.

Selection is deterministic (seeded) and greedy: from candidate `forecast_start`
positions inside the regime, shuffle then accept the next position whose window
does not overlap any previously accepted window.


In [6]:
def generate_rfp(freq: str) -> pd.DataFrame:
    rfp_cfg = cfg["rfp"]
    freq_cfg = rfp_cfg[freq]
    n_per_regime = freq_cfg["n_per_regime"]
    window_len = freq_cfg["window_len"]
    embargo = cfg["embargo_periods"]
    test_start = pd.Timestamp(cfg["splits"][freq]["test"][0])
    rng = random.Random(rfp_cfg["random_seed"])
    prefix = "d" if freq == "daily" else "w"
    dates = pd.DatetimeIndex(features[freq].index)

    rows = []
    for regime_name, (rs, re_) in rfp_cfg["regimes"].items():
        rs, re_ = pd.Timestamp(rs), pd.Timestamp(re_)
        if re_ >= test_start:
            print(f"[{freq}] skip regime {regime_name}: span end {re_.date()} >= test_start {test_start.date()}")
            continue
        in_regime_pos = [dates.get_loc(d) for d in dates[(dates >= rs) & (dates <= re_)]]
        candidates = []
        for pos in in_regime_pos:
            if pos + window_len - 1 >= len(dates):
                continue
            if pos - embargo - 1 < 0:
                continue
            forecast_end = dates[pos + window_len - 1]
            if forecast_end >= test_start:
                continue
            candidates.append(pos)
        rng.shuffle(candidates)
        chosen = []
        for c in candidates:
            if all(abs(c - x) >= window_len for x in chosen):
                chosen.append(c)
            if len(chosen) == n_per_regime:
                break
        chosen.sort()
        if len(chosen) < n_per_regime:
            print(f"[{freq}] regime {regime_name}: produced {len(chosen)} non-overlapping windows (target {n_per_regime})")
        for k, start_pos in enumerate(chosen, start=1):
            rows.append({
                "window_id": f"{prefix}_{regime_name}_{k}",
                "regime": regime_name,
                "fit_end": dates[start_pos - embargo - 1].date().isoformat(),
                "forecast_start": dates[start_pos].date().isoformat(),
                "forecast_end": dates[start_pos + window_len - 1].date().isoformat(),
            })
    return pd.DataFrame(rows)

(SPLITS / "rfp").mkdir(parents=True, exist_ok=True)
rfp_dfs = {}
for freq in ("daily", "weekly"):
    df = generate_rfp(freq)
    df.to_csv(SPLITS / "rfp" / f"{freq}_windows.csv", index=False)
    rfp_dfs[freq] = df
    print(f"\n[{freq}] {len(df)} windows")
    print(df.to_string(index=False) if len(df) else "(none)")


[daily] regime COVID: produced 3 non-overlapping windows (target 5)

[daily] 23 windows
     window_id     regime    fit_end forecast_start forecast_end
       d_GFC_1        GFC 2007-09-21     2007-09-25   2007-12-18
       d_GFC_2        GFC 2008-01-07     2008-01-09   2008-04-04
       d_GFC_3        GFC 2008-04-24     2008-04-28   2008-07-22
       d_GFC_4        GFC 2008-10-17     2008-10-21   2009-01-15
       d_GFC_5        GFC 2009-02-26     2009-03-02   2009-05-26
 d_OIL_CRASH_1  OIL_CRASH 2014-09-11     2014-09-15   2014-12-08
 d_OIL_CRASH_2  OIL_CRASH 2015-02-20     2015-02-24   2015-05-19
 d_OIL_CRASH_3  OIL_CRASH 2015-05-21     2015-05-26   2015-08-18
 d_OIL_CRASH_4  OIL_CRASH 2015-10-23     2015-10-27   2016-01-22
 d_OIL_CRASH_5  OIL_CRASH 2016-02-18     2016-02-22   2016-05-16
     d_COVID_1      COVID 2020-04-09     2020-04-14   2020-07-13
     d_COVID_2      COVID 2020-08-19     2020-08-21   2020-11-13
     d_COVID_3      COVID 2020-12-10     2020-12-14   2021-03-11
 d

## 6. Verify outputs

In [7]:
all_csvs = sorted(SPLITS.rglob("*.csv"))
print(f"Total CSVs under data/splits/: {len(all_csvs)}")
for p in all_csvs:
    print(f"  {p.relative_to(ROOT)}")


Total CSVs under data/splits/: 38
  data/splits/daily/no_exog/GOLD/test.csv
  data/splits/daily/no_exog/GOLD/train.csv
  data/splits/daily/no_exog/GOLD/val.csv
  data/splits/daily/no_exog/OIL/test.csv
  data/splits/daily/no_exog/OIL/train.csv
  data/splits/daily/no_exog/OIL/val.csv
  data/splits/daily/no_exog/SPY/test.csv
  data/splits/daily/no_exog/SPY/train.csv
  data/splits/daily/no_exog/SPY/val.csv
  data/splits/daily/with_exog/GOLD/test.csv
  data/splits/daily/with_exog/GOLD/train.csv
  data/splits/daily/with_exog/GOLD/val.csv
  data/splits/daily/with_exog/OIL/test.csv
  data/splits/daily/with_exog/OIL/train.csv
  data/splits/daily/with_exog/OIL/val.csv
  data/splits/daily/with_exog/SPY/test.csv
  data/splits/daily/with_exog/SPY/train.csv
  data/splits/daily/with_exog/SPY/val.csv
  data/splits/rfp/daily_windows.csv
  data/splits/rfp/weekly_windows.csv
  data/splits/weekly/no_exog/GOLD/test.csv
  data/splits/weekly/no_exog/GOLD/train.csv
  data/splits/weekly/no_exog/GOLD/val.csv
  

In [8]:
import itertools
checks = []
for freq, sub, target, stage in itertools.product(
    ["daily", "weekly"], ["no_exog", "with_exog"], cfg["targets"], ["train", "val", "test"]
):
    p = SPLITS / freq / sub / target / f"{stage}.csv"
    df = pd.read_csv(p, parse_dates=["date"])
    expected = cfg["splits"][freq][stage]
    checks.append({
        "freq": freq, "exog": sub, "target": target, "stage": stage,
        "rows": len(df),
        "first": df["date"].min().date() if len(df) else None,
        "last": df["date"].max().date() if len(df) else None,
        "expected": f"{expected[0]} to {expected[1]}",
    })
pd.DataFrame(checks)


,freq,exog,target,stage,rows,first,last,expected
0,daily,no_exog,SPY,train,4514,2004-01-07,2021-12-31,2004-01-01 to 2021-12-31
1,daily,no_exog,SPY,val,501,2022-01-03,2023-12-29,2022-01-01 to 2023-12-31
2,daily,no_exog,SPY,test,583,2024-01-02,2026-04-29,2024-01-01 to 2026-04-29
3,daily,no_exog,OIL,train,4514,2004-01-07,2021-12-31,2004-01-01 to 2021-12-31
4,daily,no_exog,OIL,val,501,2022-01-03,2023-12-29,2022-01-01 to 2023-12-31
5,daily,no_exog,OIL,test,583,2024-01-02,2026-04-29,2024-01-01 to 2026-04-29
6,daily,no_exog,GOLD,train,4514,2004-01-07,2021-12-31,2004-01-01 to 2021-12-31
7,daily,no_exog,GOLD,val,501,2022-01-03,2023-12-29,2022-01-01 to 2023-12-31
8,daily,no_exog,GOLD,test,583,2024-01-02,2026-04-29,2024-01-01 to 2026-04-29
9,daily,with_exog,SPY,train,4514,2004-01-07,2021-12-31,2004-01-01 to 2021-12-31


In [9]:
# Schemas
for freq in ("daily", "weekly"):
    for sub in ("no_exog", "with_exog"):
        for target in cfg["targets"]:
            p = SPLITS / freq / sub / target / "train.csv"
            cols = list(pd.read_csv(p, nrows=0).columns)
            print(f"{freq:6s}/{sub:9s}/{target:4s}: {cols}")


daily /no_exog  /SPY : ['date', 'ret', 'ret_lag1']
daily /no_exog  /OIL : ['date', 'ret', 'ret_lag1']


daily /no_exog  /GOLD: ['date', 'ret', 'ret_lag1']
daily /with_exog/SPY : ['date', 'ret', 'ret_lag1', 'OIL_ret_lag1', 'GOLD_ret_lag1', 'DXY_ret_lag1', 'vix_log_lag1']
daily /with_exog/OIL : ['date', 'ret', 'ret_lag1', 'SPY_ret_lag1', 'GOLD_ret_lag1', 'DXY_ret_lag1', 'vix_log_lag1']
daily /with_exog/GOLD: ['date', 'ret', 'ret_lag1', 'SPY_ret_lag1', 'OIL_ret_lag1', 'DXY_ret_lag1', 'vix_log_lag1']
weekly/no_exog  /SPY : ['date', 'ret', 'ret_lag1']
weekly/no_exog  /OIL : ['date', 'ret', 'ret_lag1']
weekly/no_exog  /GOLD: ['date', 'ret', 'ret_lag1']
weekly/with_exog/SPY : ['date', 'ret', 'ret_lag1', 'OIL_ret_lag1', 'GOLD_ret_lag1', 'DXY_ret_lag1', 'vix_log_lag1']
weekly/with_exog/OIL : ['date', 'ret', 'ret_lag1', 'SPY_ret_lag1', 'GOLD_ret_lag1', 'DXY_ret_lag1', 'vix_log_lag1']
weekly/with_exog/GOLD: ['date', 'ret', 'ret_lag1', 'SPY_ret_lag1', 'OIL_ret_lag1', 'DXY_ret_lag1', 'vix_log_lag1']
